# 📝 토픽 모델링 심화 과제 LV2 정답 — BERTopic 응용·토픽 대조 (강사용)

각 문제의 **모범답안 + 해설(접근법·흔한 실수·대안)** 입니다. 학생이 스스로 풀어 본 뒤 비교하도록 안내하세요.

- 경로는 정답 노트북 기준 `../../day13_토픽모델링_심화/data/`·`../../day13_토픽모델링_심화/images/` 입니다.
- 그래프 문제(2)는 자가채점이 없습니다.
- 데이터는 **두 제품군(자세밴드·선크림)** 이 섞여 있어, `product_type`(제품 종류)을 토픽이 실제 제품군을 얼마나 잘 잡아내는지 대조하는 '유사-정답'으로 씁니다.
- 임베딩 기반 결과는 하드웨어에 따라 조금 흔들릴 수 있어 **범위·형태·키워드 포함**으로 채점합니다.

아래 셀을 먼저 실행해 이 단원 라이브러리와 한국어 토크나이저를 준비하세요.

In [ ]:
# [제공 코드] 이 단원에 필요한 라이브러리와 한국어 토크나이저를 준비합니다.
import json, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from bertopic import BERTopic
from kiwipiepy import Kiwi

sns.set_theme(style='whitegrid')
import platform

# 한글 폰트 — 실행 중인 OS 에 맞춰 자동 설정
if platform.system() == 'Windows':
    KOREAN_FONT = 'Malgun Gothic'
    FONT_PATH = 'C:/Windows/Fonts/malgun.ttf'
elif platform.system() == 'Darwin':          # macOS
    KOREAN_FONT = 'AppleGothic'
    FONT_PATH = '/System/Library/Fonts/Supplemental/AppleGothic.ttf'
else:                                        # Linux (Colab 등)
    KOREAN_FONT = 'NanumGothic'
    FONT_PATH = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'

plt.rcParams['font.family'] = KOREAN_FONT
plt.rcParams['axes.unicode_minus'] = False   # 마이너스(−) 부호 깨짐 방지

# 지난 단원에서 배운 형태소 분석 — c-TF-IDF 키워드를 한국어 명사로 뽑는 토크나이저(자바가 필요없는 kiwipiepy)
kiwi = Kiwi()

# 불용어 — 11일차에서 배운 방식 그대로: 공개 일반 목록 + 이 데이터의 도메인 불용어
with open('../../day13_토픽모델링_심화/data/stopwords_ko.json', encoding='utf-8') as f:
    STOPWORDS_GENERAL = set(json.load(f))     # 11일차에서 받아 둔 공개 목록 679개

# 이 데이터에서만 무의미한 고빈도어 — 빈도표를 보고 사람이 고른다(11일차 4절)
STOPWORDS_DOMAIN = {'제품', '구매', '사용', '정말', '진짜', '완전', '그냥', '너무', '정도', '많이'}

# 반대로 일반 목록이 '여기서는 의미 있는 말'까지 지우기도 한다 — 되살릴 단어
# ('아이'·'시간' 은 일반 불용어지만, 이 리뷰에서는 '아이에게 사 준 밴드'처럼 주제를 가른다)
KEEP_WORDS = {'아이', '시간'}
KOREAN_STOPWORDS = (STOPWORDS_GENERAL | STOPWORDS_DOMAIN) - KEEP_WORDS

def korean_tokenizer(text):
    """문서에서 의미있는 명사(2글자 이상)만 골라 돌려줍니다."""
    return [t.form for t in kiwi.tokenize(str(text))
            if t.tag.startswith('NN') and len(t.form) > 1 and t.form not in KOREAN_STOPWORDS]

이어서 지난 단원에서 배운 **한국어 임베딩 모델**을 불러옵니다. 리뷰 본문은 이미 이 모델로 임베딩해 `.npy` 로 저장해 두었고, 문제 5 에서 **새 문장을 직접 임베딩**할 때 이 모델을 씁니다.

In [ ]:
# [제공 코드] 지난 단원에서 배운 한국어 임베딩 모델을 불러옵니다(문장→768차원 벡터).
emb_model = SentenceTransformer('jhgan/ko-sroberta-multitask')

# 임베딩·2D 좌표 — 저장된 파일이 있으면 그대로 쓰고, 없으면 지금 만들어 저장합니다
emb_path = '../../day13_토픽모델링_심화/data/reviews_mixed_embeddings.npy'
umap_path = '../../day13_토픽모델링_심화/data/reviews_mixed_umap2d.npy'
if not os.path.exists(emb_path):
    print('저장된 임베딩이 없어 지금 만듭니다 — 수 분 걸릴 수 있어요')
    texts = pd.read_csv('../../day13_토픽모델링_심화/data/reviews_mixed.csv')['text'].astype(str).tolist()
    np.save(emb_path, emb_model.encode(texts, show_progress_bar=False))
if not os.path.exists(umap_path):
    print('저장된 2차원 좌표가 없어 지금 만듭니다')
    np.save(umap_path, UMAP(n_components=2, n_neighbors=15, min_dist=0.1,
                            metric='cosine', random_state=42).fit_transform(np.load(emb_path)))

## 데이터 살펴보기 — 먼저 데이터를 이해합니다
새 데이터셋이니 분석 전에 먼저 파악합니다. `head()`·`info()` 와 함께, 이번엔 **제품 종류(`product_type`)의 분포**도 확인합니다(자세밴드·선크림이 각각 몇 개인지). (아래 셀은 실행만 하면 됩니다.)

In [ ]:
# [제공 코드] 데이터를 먼저 살펴봅니다 — 앞부분·구조·제품 종류 분포
reviews = pd.read_csv('../../day13_토픽모델링_심화/data/reviews_mixed.csv')
print("행·열 크기:", reviews.shape)
print("\n[앞 5행] head()"); display(reviews.head())
print("\n[열·자료형·결측] info()"); reviews.info()
print("\n[제품 종류 분포] value_counts()"); display(reviews["product_type"].value_counts())

## 1. 데이터 살펴보기 (서술형)
**배경**: 위 `데이터 살펴보기` 출력을 보고 이 혼합 리뷰 데이터에 대해 알게 된 사실을 정리해 보세요.

**요구사항**:
- 아래 서술 셀에 **관찰 2~3가지**를 적으세요.
- 그리고 **예측**을 하나 적으세요: 두 제품군 리뷰에 각각 **어떤 단어가 많이 나올 것 같은지**. 문제 3 에서 BERTopic 이 실제로 뽑은 키워드와 대조해 볼 것입니다.
- 예: 전체 리뷰 수, 자세밴드·선크림 각각의 개수(어느 쪽이 더 많은지), 두 제품군의 리뷰 내용이 어떻게 다를지 등.
- 정답은 하나가 아닙니다. 출력에서 실제로 확인되는 사실이면 됩니다.

> 이 문제는 자가채점(assert)이 없습니다. 정답 노트북의 모범 서술과 비교해 보세요.

**모범 서술 (예시 — 정답은 여럿)**

- 전체 리뷰는 1445개이고, 열은 `product_type`(제품 종류)과 `text`(리뷰 본문)다. 결측치는 없다.
- 제품 종류는 **자세밴드 900개, 선크림 545개**로 자세밴드 리뷰가 더 많다.
- 두 제품군은 어휘가 크게 다를 것이다 — 자세밴드는 '착용·사이즈·어깨·자세', 선크림은 '크림·피부·로션·발림' 같은 단어가 많을 것이다. 그래서 임베딩만으로도 두 군이 잘 갈릴 가능성이 높다.

## 2. 제품 종류별 색으로 보는 임베딩 지도 (조합)
**배경**: LV1 에서는 한 색으로 UMAP 산점도를 그렸습니다. 이번엔 **임베딩 시각화 + 라벨 색칠**을 조합해, 2D 지도 위에서 자세밴드와 선크림이 **색으로 나뉘는지** 확인합니다. 실제로 두 군이 멀리 떨어져 있다면, 임베딩이 제품군의 의미 차이를 잘 담은 것입니다.

**요구사항**:
- `../../day13_토픽모델링_심화/data/reviews_mixed.csv` 를 `reviews` 에, `../../day13_토픽모델링_심화/data/reviews_mixed_umap2d.npy` 를 `coords` 에 담으세요.
- `plt.figure(figsize=(7, 6))` 로 새 그림을 연 뒤, `product_type` 의 **종류별로 반복**하며 그 종류에 해당하는 점만 `plt.scatter(...)` 로 그리고 `label=종류` 를 주세요(두 종류가 서로 다른 색).
- 범례(`plt.legend()`)·제목·축 이름을 달고 `plt.show()` 로 보여 주세요.

**예시**: 아래 완성 그래프처럼 자세밴드·선크림이 서로 다른 색의 두 덩어리로 나뉘면 됩니다. (이 문제는 자가채점이 없습니다.)
<details><summary>힌트</summary>

```text
접근방법:
- 2D 좌표를 불러오고, 제품 종류의 고유값마다 그 종류의 점만 골라 색을 달리해 산점도를 겹쳐 그린다.

세부구현:
1. csv 를 reviews 에, 2D 좌표를 coords 에 담는다
2. plt.figure 로 새 그림을 연다
3. product_type 의 고유값을 돌면서, 그 종류인 행만 골라(불리언 마스크) 그 위치의 좌표로 scatter 를 그린다(label 지정)
4. 범례·제목·축 이름을 달고 plt.show 로 보여 준다
```

</details>

> **완성 그래프(정답)** — 아래 그림과 같은 모양이 나오도록 그려 보세요.

<img src="../../day13_토픽모델링_심화/images/과제/lv2_q2_umap_type.png" width="560">

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

reviews = pd.read_csv('../../day13_토픽모델링_심화/data/reviews_mixed.csv')
coords = np.load('../../day13_토픽모델링_심화/data/reviews_mixed_umap2d.npy')
plt.figure(figsize=(7, 6))
for name in reviews['product_type'].unique():
    mask = (reviews['product_type'] == name).values
    plt.scatter(coords[mask, 0], coords[mask, 1], s=10, alpha=0.5, label=name)
plt.title('자세밴드+선크림 리뷰 임베딩의 2D 지도 (제품 종류별)')
plt.xlabel('UMAP 1')
plt.ylabel('UMAP 2')
plt.legend()
plt.show()

### 해설 — 문제 2
- **접근법**: 종류별로 마스크(`product_type == name`)를 만들어 scatter 를 겹쳐 그리면 자동으로 색과 범례가 분리됩니다. 두 덩어리가 뚜렷이 갈리면 임베딩이 제품군 차이를 잘 담았다는 신호이고, 다음 문제의 BERTopic 이 이 구조를 토픽 키워드로 확인해 줍니다.
- **흔한 실수**: 불리언 마스크를 넘길 때 `.values` 로 numpy 배열을 쓰면 좌표 배열 인덱싱과 안전하게 맞습니다(판다스 인덱스 정렬 혼동 방지). 또 `plt.figure()` 로 새 그림을 여세요.

## 3. BERTopic 으로 두 제품군의 주제 뽑기 (조합)
**배경**: **표준 구성 + 학습 + 키워드 해석**을 조합합니다. 혼합 리뷰에 BERTopic 을 돌리면, 두 제품군이 서로 다른 토픽으로 갈리는지 키워드로 확인할 수 있습니다. 이번엔 리뷰가 많으니 `min_topic_size=25` 로 굵직한 토픽을 뽑습니다.

**요구사항**:
- `../../day13_토픽모델링_심화/data/reviews_mixed.csv` 의 `text` 를 리스트로 `docs` 에, 임베딩을 `np.load` 로 `emb` 에 담으세요.
- 아래 표준 구성으로 BERTopic 을 만드세요(`min_cluster_size=25`, 제공 코드의 `emb_model`·`korean_tokenizer` 사용):
```
umap_model = UMAP(n_components=5, n_neighbors=15, min_dist=0.0, metric='cosine', random_state=42)
hdbscan_model = HDBSCAN(min_cluster_size=25, metric='euclidean', prediction_data=True)
vectorizer_model = CountVectorizer(tokenizer=korean_tokenizer, max_df=0.85)
topic_model = BERTopic(embedding_model=emb_model, umap_model=umap_model,
                       hdbscan_model=hdbscan_model, vectorizer_model=vectorizer_model,
                    language='multilingual', verbose=False)
```
- `topic_model.fit_transform(docs, embeddings=emb)` 의 반환값을 `topics, probs` 에 담으세요.
- `topic_model.get_topic_info()` 를 `topic_info` 에 담고, 노이즈(−1)를 뺀 **모든 토픽의 키워드**를 모아 하나의 리스트 `all_keywords` 에 담으세요. 토픽별로 나누지 말고 **단어 문자열만 한 줄로 이어 붙인 평평한 리스트**여야 합니다(예: `['착용', '사이즈', …, '크림', '피부', …]`).

**예시**
```
노이즈(-1) 제외 토픽 수는 대략 2~8개 (이 데이터·이 설정에서는 2개가 나옵니다)
all_keywords 안에 '자세/어깨/교정' 같은 밴드 단어와 '썬크림/자외선/차단' 같은 선크림 단어가 함께 등장
```
<details><summary>힌트</summary>

```text
접근방법:
- 표준 구성(min_cluster_size=25)으로 BERTopic 을 학습한다.
- 노이즈(-1)를 뺀 각 토픽의 키워드에서 단어만 뽑아 하나의 리스트로 합친다.

세부구현:
1. csv 의 text 를 docs 에, 임베딩을 emb 에 담는다
2. 표준 구성으로 topic_model 을 만들고 fit_transform(docs, embeddings=emb) 한다
3. get_topic_info 를 topic_info 에 담는다
4. Topic 이 -1 이 아닌 각 토픽마다 get_topic(tid) 의 (단어, 점수)에서 단어만 골라 all_keywords 에 모은다
```

</details>

In [ ]:
docs = pd.read_csv('../../day13_토픽모델링_심화/data/reviews_mixed.csv')['text'].tolist()
emb = np.load('../../day13_토픽모델링_심화/data/reviews_mixed_embeddings.npy')
umap_model = UMAP(n_components=5, n_neighbors=15, min_dist=0.0, metric='cosine', random_state=42)
hdbscan_model = HDBSCAN(min_cluster_size=25, metric='euclidean', prediction_data=True)
vectorizer_model = CountVectorizer(tokenizer=korean_tokenizer, max_df=0.85)
topic_model = BERTopic(embedding_model=emb_model, umap_model=umap_model,
                       hdbscan_model=hdbscan_model, vectorizer_model=vectorizer_model,
                    language='multilingual', verbose=False)
topics, probs = topic_model.fit_transform(docs, embeddings=emb)
topic_info = topic_model.get_topic_info()
display(topic_info)

all_keywords = []
for tid in topic_info.loc[topic_info['Topic'] != -1, 'Topic']:
    all_keywords += [word for word, score in topic_model.get_topic(tid)]
print("전체 키워드 일부:", all_keywords[:20])

In [ ]:
# [자가채점]
n_topics = int((topic_info['Topic'] != -1).sum())
assert 2 <= n_topics <= 8
assert any(w in all_keywords for w in ['자세', '어깨', '교정', '허리', '밴드', '착용감', '통증']), '밴드 계열 단어가 안 보입니다 — all_keywords 가 단어 문자열만 담은 평평한 리스트인지 확인하세요'
assert any(w in all_keywords for w in ['썬크림', '자외선', '차단', '성분', '수분', '보습', '메이크업']), '선크림 계열 단어가 안 보입니다 — all_keywords 가 단어 문자열만 담은 평평한 리스트인지 확인하세요'
print("✅ 문제3 통과! (토픽 수:", n_topics, ")")

### 해설 — 문제 3
- **접근법**: `min_cluster_size` 를 25 로 키우면 잔가지 토픽이 줄고, 두 제품군에 해당하는 굵은 토픽만 남습니다. 실제로 토픽은 딱 2개가 나오는데, 하나는 '자세·어깨·교정·허리'(자세밴드 936건), 다른 하나는 '썬크림·자외선·차단·성분'(선크림 509건)입니다 — 임베딩·토픽 모델이 라벨 없이도 제품군을 갈라낸 셈입니다. 이 설정에서는 노이즈(−1)도 한 건도 나오지 않습니다.
- **흔한 실수**: 토픽 번호(0, 1 …)는 실행 환경에 따라 순서가 바뀔 수 있어, 특정 번호의 키워드만 확인하면 불안정합니다. **모든 토픽의 키워드를 합쳐** 두 계열 단어가 있는지 보는 것이 안전합니다.
- **대안**: 토픽이 너무 뭉뚱그려지면 `min_cluster_size` 를 낮춰 더 잘게 나눌 수 있습니다.

## 4. 토픽의 대표 문서를 제품 종류와 대조 (조합)
**배경**: **대표 문서 조회 + 라벨 대조**를 조합합니다. 각 토픽의 대표 문서가 실제로 어느 제품군의 리뷰인지 확인하면, 토픽이 제품군과 맞물리는지 눈으로 검증할 수 있습니다.

**요구사항**:
- 문제 3 의 `topic_model`·`topics` 와 위 `데이터 살펴보기` 셀에서 불러온 `reviews` 를 사용하세요. `reviews` 가 없으면 `../../day13_토픽모델링_심화/data/reviews_mixed.csv` 를 다시 불러오세요.
- 노이즈(−1)를 뺀 토픽 중 **번호가 가장 작은 것**을 `first_topic` 에 담으세요.
- 그 토픽에 속한 리뷰들의 `product_type` 분포를 세어 `dist` 에 담으세요(그 토픽 문서들의 제품 종류 `value_counts()`). `topics` 는 각 문서의 토픽 번호 리스트이니, `np.array(topics) == first_topic` 마스크로 해당 리뷰를 고릅니다.

**예시**
```
first_topic  →  0
dist         →  그 토픽 문서들의 제품 종류 개수(대개 한 제품군이 압도적)
```
<details><summary>힌트</summary>

```text
접근방법:
- 노이즈를 뺀 가장 작은 토픽 번호를 고르고, 그 토픽에 배정된 문서만 골라 제품 종류 분포를 센다.

세부구현:
1. topic_info 에서 Topic 이 -1 이 아닌 값 중 가장 작은 것을 first_topic 에 담는다
2. topics 를 numpy 배열로 만들어 first_topic 과 같은 위치의 마스크를 만든다
3. reviews 의 product_type 을 그 마스크로 골라 value_counts 로 dist 에 담는다
```

</details>

In [ ]:
first_topic = sorted(topic_info.loc[topic_info['Topic'] != -1, 'Topic'])[0]
mask = np.array(topics) == first_topic
dist = reviews.loc[mask, 'product_type'].value_counts()
print(f"토픽 {first_topic} 의 제품 종류 분포:")
display(dist)

In [ ]:
# [자가채점]
assert dist.sum() >= 1
assert dist.sum() == int((np.array(topics) == first_topic).sum())
print("✅ 문제4 통과!")

### 해설 — 문제 4
- **접근법**: `topics` 는 각 문서가 어느 토픽에 갔는지 담은 배열입니다. 마스크로 한 토픽의 문서만 골라 `product_type` 을 세면, 그 토픽이 어느 제품군에 치우쳤는지 바로 보입니다. 보통 한 제품군이 압도적이라, 토픽이 제품군과 잘 맞물린다는 것을 확인할 수 있습니다.
- **흔한 실수**: `topics` 는 리스트라 불리언 마스크가 안 되니 `np.array(topics)` 로 바꿔야 합니다. 또 `reviews` 의 행 순서가 `docs`·`topics` 와 같아야 대조가 맞습니다(같은 csv 순서이므로 일치).
- **대안**: 문제 6 의 교차표로 모든 토픽×제품군을 한 번에 대조하면 더 큰 그림이 보입니다.

## 5. 새 리뷰는 어느 주제일까 — `transform` 으로 토픽 배정 (조합)
**배경**: 토픽 모델의 진짜 쓸모는 **새로 들어온 리뷰를 자동 분류**하는 것입니다. 학습된 BERTopic 은 `transform(새 문서)` 으로 **이미 찾아 둔 토픽 중 어디에 속하는지** 알려 줍니다(다시 학습하지 않습니다). **직접 임베딩(encode) + transform + 키워드 확인**을 조합해, 새 리뷰 두 개를 배정하고 그 배정이 말이 되는지 토픽 키워드로 검증해 봅시다.

**요구사항**:
- 문제 3 의 `topic_model` 을 그대로 사용하세요(다시 `fit` 하지 마세요).
- 아래 두 새 문장을 `emb_model.encode(new_reviews)` 로 임베딩해 `new_emb`(2×768)에 담으세요.
```
new_reviews = ["어깨가 굽어서 자세교정 하려고 밴드를 착용했어요",
               "백탁 없이 촉촉하게 발리는 선크림이라 만족합니다"]
```
- `topic_model.transform(new_reviews, embeddings=new_emb)` 의 반환값을 `new_topics, new_probs` 에 담으세요. `new_topics` 는 각 새 문장에 배정된 **토픽 번호** 입니다(−1 이면 어느 토픽에도 안 붙음).
- 배정된 토픽의 키워드를 `topic_model.get_topic(번호)` 로 출력해, 밴드 문장과 선크림 문장이 각각 **어울리는 토픽**에 갔는지 눈으로 확인하세요.

**예시**
```
len(new_topics)            →  2
new_topics[0] != new_topics[1]  →  True   (밴드 문장과 선크림 문장은 서로 다른 토픽)
밴드 문장이 간 토픽의 키워드에 '자세·어깨' 가, 선크림 문장 쪽엔 '썬크림·자외선' 이 보입니다
```
<details><summary>힌트</summary>

```text
접근방법:
- 이미 학습된 토픽 모델에 새 문서를 넣으면(transform) 기존 토픽 중 하나로 배정된다. 재학습은 하지 않는다.
- 학습 때와 같은 임베딩 모델로 새 문장을 벡터화해 embeddings 인자로 함께 넘긴다.

세부구현:
1. new_reviews 두 문장을 emb_model.encode 로 new_emb 에 담는다
2. topic_model.transform 에 (new_reviews, embeddings=new_emb) 을 넘겨 new_topics, new_probs 에 담는다
3. 배정된 각 토픽 번호로 get_topic 을 호출해 키워드를 출력하고 문장 내용과 맞는지 확인한다
```

</details>

In [ ]:
new_reviews = ["어깨가 굽어서 자세교정 하려고 밴드를 착용했어요",
               "백탁 없이 촉촉하게 발리는 선크림이라 만족합니다"]
new_emb = emb_model.encode(new_reviews)
new_topics, new_probs = topic_model.transform(new_reviews, embeddings=new_emb)
for sent, tid in zip(new_reviews, new_topics):
    words = [w for w, _ in topic_model.get_topic(int(tid))] if int(tid) != -1 else []
    print(f"{sent}\n  → 토픽 {int(tid)} | 키워드: {words[:6]}")

In [ ]:
# [자가채점]
assert len(new_topics) == 2
_t0, _t1 = int(new_topics[0]), int(new_topics[1])
assert _t0 != -1 and _t1 != -1, '두 문장 모두 토픽에 배정돼야 합니다(재학습 말고 transform 을 쓰세요)'
assert _t0 != _t1, '밴드 문장과 선크림 문장은 서로 다른 토픽으로 갈려야 합니다'
# 배정이 우연이 아닌지 — 각 토픽의 키워드가 문장 내용과 맞는지 대조
_kw0 = [w for w, _ in topic_model.get_topic(_t0)]
_kw1 = [w for w, _ in topic_model.get_topic(_t1)]
assert any(w in _kw0 for w in ['자세', '어깨', '교정', '허리', '밴드', '착용감']), '밴드 문장이 밴드 토픽에 가지 않았습니다'
assert any(w in _kw1 for w in ['썬크림', '자외선', '차단', '성분', '수분', '보습']), '선크림 문장이 선크림 토픽에 가지 않았습니다'
print("✅ 문제5 통과! (배정 토픽:", _t0, ",", _t1, ")")

### 해설 — 문제 5
- **접근법**: `fit_transform` 은 **토픽을 찾는** 학습이고, `transform` 은 **이미 찾은 토픽에 새 문서를 배정**하는 추론입니다. 안에서는 학습 때 만든 UMAP 으로 새 임베딩을 같은 공간에 투영한 뒤, HDBSCAN 의 `approximate_predict` 로 가장 가까운 토픽을 고릅니다(그래서 모델을 만들 때 `prediction_data=True` 가 필요했습니다). 실측 결과 밴드 문장은 '자세·어깨·교정' 토픽, 선크림 문장은 '썬크림·자외선·차단' 토픽으로 각각 갈립니다.
- **흔한 실수**: 새 문서를 넣겠다고 `fit_transform` 을 다시 부르면 **토픽 자체가 새로 학습돼** 앞 문제들의 토픽 번호와 어긋납니다. 추론은 반드시 `transform` 입니다. 또 `embeddings=` 를 넘기지 않으면 모델이 문장을 다시 임베딩하느라 느려집니다(결과는 같습니다).
- **대안**: 어느 토픽에도 안 붙어 −1 이 나오면, `topic_model.approximate_distribution(new_reviews)` 로 토픽별 비중을 구해 가장 큰 토픽에 배정할 수 있습니다.

## 6. 토픽 × 제품 종류 교차표 (조합)
**배경**: **토픽 배정 + 교차 집계**를 조합해, 어느 토픽이 어느 제품군에 몰리는지 **표 하나로** 봅니다. `pd.crosstab` 은 두 범주를 행·열로 교차해 개수를 세 줍니다.

**요구사항**:
- 문제 3 의 `topics` 와 위 `데이터 살펴보기` 셀에서 불러온 `reviews` 를 사용하세요.
- `pd.crosstab(topics, reviews['product_type'])` 로 **행=토픽 번호, 열=제품 종류** 인 교차표를 만들어 `ct` 에 담고 `display(ct)` 로 확인하세요. (`topics` 는 각 문서의 토픽 번호 리스트입니다.)

**예시**
```
ct.shape       →  (행 = 배정된 토픽 종류 수, 2)   # 열은 제품 종류 2개
display(ct)    →  토픽별로 자세밴드·선크림이 몇 개인지
* 노이즈(-1)가 있으면 -1 행도 함께 나옵니다. 이 설정에서는 노이즈가 없어 -1 행이 없습니다.
```
<details><summary>힌트</summary>

```text
접근방법:
- 각 문서의 토픽 번호와 제품 종류를 crosstab 으로 교차 집계한다(행=토픽, 열=제품 종류).

세부구현:
1. pd.crosstab 에 (topics, reviews['product_type']) 을 넘겨 ct 에 담는다
2. display 로 표를 확인한다
```

</details>

In [ ]:
ct = pd.crosstab(topics, reviews['product_type'])
display(ct)

In [ ]:
# [자가채점]
assert ct.shape[1] == 2          # 제품 종류 2개(자세밴드·선크림)
assert ct.shape[0] >= 2          # 배정된 토픽이 2종류 이상(노이즈 -1 이 있으면 그 행도 포함)
assert int(ct.values.sum()) == len(reviews)   # 모든 리뷰가 집계됨
print("✅ 문제6 통과!")

### 해설 — 문제 6
- **접근법**: `crosstab` 은 두 범주형의 조합별 개수를 표로 줍니다. 여기서는 토픽과 제품 종류를 교차해, 각 토픽이 자세밴드·선크림 중 어디에 몰리는지 한눈에 보여 줍니다. 실제로 토픽 0 은 자세밴드 899·선크림 37, 토픽 1 은 선크림 508·자세밴드 1 로 **한 제품군에 뚜렷이 쏠려** 토픽이 제품군을 잘 반영했음이 표 하나로 확인됩니다.
- **참고**: 노이즈(−1)가 있는 모델이라면 `-1` 행도 함께 나옵니다. 이 설정(`min_cluster_size=25`)에서는 노이즈가 한 건도 없어 `-1` 행이 아예 없습니다 — 노이즈 유무는 모델 설정과 데이터에 따라 달라집니다.
- **흔한 실수**: `topics` 와 `reviews` 의 행 순서(길이)가 같아야 합니다 — 같은 csv 순서이므로 일치합니다. 순서가 어긋난 데이터를 넣으면 교차표가 엉킵니다.
- **대안**: 개수 대신 비율로 보려면 `pd.crosstab(..., normalize='index')` 로 행 기준 비율을 볼 수 있습니다.

## 7. 굵게 vs 잘게 — 토픽 수가 시각화를 어떻게 바꾸나 (조합)
**배경**: **`min_cluster_size` 조절 + 내장 시각화**를 조합합니다. 문제 3 의 모델은 `min_cluster_size=25` 로 **굵게** 뽑아 토픽이 **2개뿐**입니다. 토픽이 2개면 히트맵은 2×2, 계층도는 잎이 2개라 **볼 것이 거의 없습니다**. 같은 리뷰를 `min_cluster_size=10` 으로 **잘게** 다시 뽑아 같은 그림 3종을 그려 보고, **토픽 수가 그림의 쓸모를 어떻게 바꾸는지** 눈으로 확인합니다.

**요구사항**:
- (굵은 모델) 문제 3 의 `topic_model` 로 세 그림을 만들어 각각 `fig_bar`·`fig_heat`·`fig_tree` 에 담으세요 — `visualize_barchart(top_n_topics=4)` · `visualize_heatmap()` · `visualize_hierarchy()`.
- (잘게 뽑은 모델) 문제 3 과 **똑같은 표준 구성인데 `min_cluster_size` 만 10** 으로 준 모델을 새로 만들어 `fine_model` 에 담고 `fine_model.fit_transform(docs, embeddings=emb)` 로 학습하세요. 학습된 토픽 수(노이즈 −1 제외)를 `n_fine` 에 정수로 담으세요.
- `fine_model` 로도 같은 세 그림을 만들어 `fig_bar2`·`fig_heat2`·`fig_tree2` 에 담으세요.
- 여섯 그림을 화면에 띄우세요. plotly Figure 는 **셀 마지막 줄에 변수 이름만** 두면 렌더됩니다. 한 셀에는 **그림 하나씩** — 여러 개를 한 셀에 두면 마지막 하나만 나옵니다.

**예시**
```
n_fine                  ->  6 안팎 (굵은 모델의 2개보다 많으면 통과)
fig_bar   / fig_bar2    ->  키워드 막대 2개 / 4개
fig_heat  / fig_heat2   ->  2x2 히트맵 / n_fine x n_fine 히트맵(대각선이 가장 진함)
fig_tree  / fig_tree2   ->  잎 2개짜리 트리 / 잎이 n_fine 개인 계층 트리
```
<details><summary>힌트</summary>

```text
접근방법:
- 이미 학습된 굵은 모델로 그림 3종을 만든다.
- 문제 3 과 같은 구성에서 min_cluster_size 만 10 으로 바꾼 모델을 새로 학습해 그림 3종을 더 만든다.
- 한 셀에서 여러 그림을 띄우려면 그림마다 show 메서드를 부른다.

세부구현:
1. 굵은 모델의 막대(top_n_topics=4)·히트맵·계층도를 fig_bar·fig_heat·fig_tree 에 담는다
2. UMAP·CountVectorizer 는 문제 3 과 같게, HDBSCAN 의 min_cluster_size 만 10 으로 준 fine_model 을 만든다
3. fine_model 을 docs 와 embeddings=emb 로 학습한다
4. get_topic_info() 에서 Topic 이 -1 이 아닌 행 수를 n_fine 에 담는다
5. fine_model 로 같은 3종을 fig_bar2·fig_heat2·fig_tree2 에 담는다
6. 그림은 한 셀에 하나씩 두고, 셀 마지막 줄에 변수 이름만 남긴다
```

</details>

In [ ]:
# (1) 굵은 모델 — 문제 3 의 topic_model (토픽 2개)
fig_bar = topic_model.visualize_barchart(top_n_topics=4)
fig_heat = topic_model.visualize_heatmap()
fig_tree = topic_model.visualize_hierarchy()

# (2) 잘게 뽑은 모델 — min_cluster_size 만 10 으로
fine_umap = UMAP(n_components=5, n_neighbors=15, min_dist=0.0, metric='cosine', random_state=42)
fine_hdbscan = HDBSCAN(min_cluster_size=10, metric='euclidean',
                       prediction_data=True)
fine_vectorizer = CountVectorizer(tokenizer=korean_tokenizer, max_df=0.85)
fine_model = BERTopic(embedding_model=emb_model, umap_model=fine_umap,
                      hdbscan_model=fine_hdbscan, vectorizer_model=fine_vectorizer,
                      language='multilingual', verbose=False)
fine_model.fit_transform(docs, embeddings=emb)
n_fine = int((fine_model.get_topic_info()['Topic'] != -1).sum())

fig_bar2 = fine_model.visualize_barchart(top_n_topics=4)
fig_heat2 = fine_model.visualize_heatmap()
fig_tree2 = fine_model.visualize_hierarchy()

n_coarse = int((topic_model.get_topic_info()['Topic'] != -1).sum())
print(f"굵은 모델 토픽 {n_coarse}개  vs  잘게 뽑은 모델 토픽 {n_fine}개")
print('토픽이 2개뿐이면 히트맵·계층도는 볼 것이 거의 없다 — 토픽이 늘어야 이 그림들이 쓸모를 갖는다')

In [ ]:
# [자가채점]
import plotly.graph_objects as go
for _f in [fig_bar, fig_heat, fig_tree, fig_bar2, fig_heat2, fig_tree2]:
    assert isinstance(_f, go.Figure), '여섯 그림 모두 plotly Figure 여야 합니다'
# 잘게 뽑은 모델은 굵은 모델보다 토픽이 많아야 한다
_n_coarse = int((topic_model.get_topic_info()['Topic'] != -1).sum())
assert n_fine == int((fine_model.get_topic_info()['Topic'] != -1).sum()), 'n_fine 은 fine_model 에서 실제로 센 값이어야 합니다'
assert n_fine > _n_coarse, 'min_cluster_size 를 10 으로 낮춘 모델이 더 많은 토픽을 찾아야 합니다'
assert 3 <= n_fine <= 12
# 그림이 실제로 그 모델에서 나왔는지 — 막대 개수가 각 모델의 토픽 수를 따라간다
assert len(fig_bar.data) == min(4, _n_coarse), 'fig_bar 는 굵은 모델로 top_n_topics=4 로 그려야 합니다'
assert len(fig_bar2.data) == min(4, n_fine), 'fig_bar2 는 fine_model 로 top_n_topics=4 로 그려야 합니다'
print("✅ 문제7 통과! (굵은 모델", _n_coarse, "토픽 / 잘게 뽑은 모델", n_fine, "토픽)")

### 해설 — 문제 7
- **접근법**: 세 그림은 **같은 토픽 집합을 다른 각도로** 봅니다. 막대는 "이 토픽이 무슨 말로 이루어졌나", 히트맵은 "어떤 토픽끼리 겹치나", 계층도는 "합친다면 어느 것부터 합칠까"를 답합니다. 그런데 **뒤 두 그림은 토픽이 여럿일 때만 의미**가 있습니다 — 굵은 모델은 토픽이 2개라 히트맵이 2×2, 계층도는 한 번 합치면 끝입니다. `min_cluster_size` 를 10 으로 낮추면 토픽이 6개로 늘어 비로소 "어떤 토픽끼리 가까운가"를 읽을 수 있고, 가까이 붙은 토픽이 `reduce_topics`·`merge_topics` 의 후보가 됩니다.
- **무엇을 배웠나**: 시각화가 시시하다면 그림 탓이 아니라 **토픽을 너무 굵게 잡은 탓**일 수 있습니다. `min_cluster_size` 는 결과뿐 아니라 **결과를 읽는 도구의 쓸모까지** 바꿉니다.
- **흔한 실수**: 1) 그림이 안 보인다고 당황하는 것 — BERTopic 시각화는 matplotlib 이 아니라 **plotly Figure** 라서 `plt.show()` 로는 안 나옵니다. 셀 **마지막 줄에 변수 이름만** 두세요. 2) 한 셀에 그림을 여러 개 늘어놓는 것 — **마지막 하나만** 렌더됩니다. 그림당 셀 하나로 나누세요. 3) `fine_model` 을 만들 때 `random_state=42` 나 `max_df=0.85` 를 빠뜨리는 것.
- **대안**: 문서 하나하나를 점으로 보고 싶으면 `visualize_documents(docs, embeddings=emb)` 를 씁니다(교안 6절 따라하기). 토픽이 많을 땐 `top_n_topics` 로 잘라서 봐야 읽힙니다.

## 8. 키워드 뽑는 방식 3종 비교하기 (조합)
**배경**: 문제 3 의 키워드는 **c-TF-IDF**(그 토픽에서 유난히 자주 나오는 단어)로 뽑힌 것입니다. BERTopic 은 `representation_model` 자리에 **다른 방식**을 끼울 수 있습니다(교안 8절). **KeyBERT 방식**은 빈도가 아니라 **문서 임베딩과 뜻이 가까운** 단어를, **MMR** 은 후보 중 **서로 겹치지 않는** 단어를 고릅니다. 세 방식을 나란히 놓고 **어느 쪽이 이 데이터에 맞는지 직접 판단**해 보세요.

**요구사항**:
- 문제 3 과 **똑같은 표준 구성**(`min_cluster_size=25`)에 `representation_model` 만 바꾼 모델 **두 개**를 만들어 각각 `kb_model`(KeyBERT)·`mmr_model`(MMR) 에 담고 학습하세요.
```
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance
#   KeyBERTInspired()                        -> 뜻이 가까운 단어
#   MaximalMarginalRelevance(diversity=0.3)  -> 서로 안 겹치는 단어
```
- 토픽마다 **세 방식의 키워드 5개씩**을 나란히 담은 DataFrame 을 `compare_rep` 에 만들어 `display` 하세요. 컬럼은 `topic_id`·`ctfidf`·`keybert`·`mmr` 로 하세요.
- 두 모델의 **문서별 토픽 배정이 같은지** 비교해 `same_assign` 에 담으세요(True/False).

**예시**
```
compare_rep  ->  topic_id | ctfidf | keybert | mmr   (세 열의 단어가 서로 다르다)
same_assign  ->  True   (표현만 바뀌고 군집화는 그대로)
```
<details><summary>힌트</summary>

```text
접근방법:
- 문제 3 의 구성을 그대로 만들되 representation_model 인자만 더한다.
- 두 모델에서 같은 토픽 번호의 키워드를 각각 뽑아 한 행으로 묶는다.
- 문서별 배정은 각 모델의 topics_ 속성을 통째로 비교하면 된다.

세부구현:
1. KeyBERTInspired 를 import 한다
2. 문제 3 과 같은 표준 구성에 representation_model 을 더해 kb_model 을 만들고 학습한다
3. 노이즈(-1)를 뺀 각 토픽마다 두 모델의 get_topic 앞 5개 단어를 모아 표를 만든다
4. topic_model.topics_ 와 kb_model.topics_ 를 == 로 비교해 same_assign 에 담는다
```

</details>

In [ ]:
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance

kb_umap = UMAP(n_components=5, n_neighbors=15, min_dist=0.0, metric='cosine', random_state=42)
kb_hdbscan = HDBSCAN(min_cluster_size=25, metric='euclidean',
                     prediction_data=True)
kb_vectorizer = CountVectorizer(tokenizer=korean_tokenizer, max_df=0.85)
kb_model = BERTopic(embedding_model=emb_model, umap_model=kb_umap,
                    hdbscan_model=kb_hdbscan, vectorizer_model=kb_vectorizer,
                    representation_model=KeyBERTInspired(),
                    language='multilingual', verbose=False)
kb_model.fit_transform(docs, embeddings=emb)

mmr_umap = UMAP(n_components=5, n_neighbors=15, min_dist=0.0, metric='cosine', random_state=42)
mmr_hdbscan = HDBSCAN(min_cluster_size=25, metric='euclidean', prediction_data=True)
mmr_vectorizer = CountVectorizer(tokenizer=korean_tokenizer, max_df=0.85)
mmr_model = BERTopic(embedding_model=emb_model, umap_model=mmr_umap,
                     hdbscan_model=mmr_hdbscan, vectorizer_model=mmr_vectorizer,
                     representation_model=MaximalMarginalRelevance(diversity=0.3),
                     language='multilingual', verbose=False)
mmr_model.fit_transform(docs, embeddings=emb)

rows = []
for tid in topic_info.loc[topic_info['Topic'] != -1, 'Topic']:
    rows.append({'topic_id': int(tid),
                 'ctfidf': ', '.join(w for w, _ in topic_model.get_topic(tid)[:5]),
                 'keybert': ', '.join(w for w, _ in kb_model.get_topic(tid)[:5]),
                 'mmr': ', '.join(w for w, _ in mmr_model.get_topic(tid)[:5])})
compare_rep = pd.DataFrame(rows)
display(compare_rep)

same_assign = (topic_model.topics_ == kb_model.topics_) and (topic_model.topics_ == mmr_model.topics_)
print('문서별 토픽 배정이 같은가:', same_assign)
print('-> 표현 모델은 이미 묶인 토픽에 단어만 새로 붙인다. 군집화는 건드리지 않는다')

In [ ]:
# [자가채점]
_n = int((topic_info['Topic'] != -1).sum())
assert set(compare_rep.columns) == {'topic_id', 'ctfidf', 'keybert', 'mmr'}, '컬럼은 topic_id·ctfidf·keybert·mmr 여야 합니다'
assert len(compare_rep) == _n, f'노이즈를 뺀 토픽 수({_n})만큼 행이 있어야 합니다'
assert (compare_rep['ctfidf'] != compare_rep['keybert']).any(), 'KeyBERT 키워드가 기본과 전부 같습니다 — representation_model 이 끼워졌는지 확인하세요'
assert (compare_rep['ctfidf'] != compare_rep['mmr']).any(), 'MMR 키워드가 기본과 전부 같습니다 — diversity 를 준 모델인지 확인하세요'
assert same_assign is True or same_assign == True, '표현 모델만 바꿨으면 문서별 배정은 같아야 합니다'
print('✅ 문제8 통과!')

### 해설 — 문제 8
- **접근법**: `representation_model` 은 UMAP·HDBSCAN·CountVectorizer 와 **같은 자리의 부품**입니다. 하나만 갈아 끼우면 나머지는 그대로 돌아갑니다.
- **핵심 확인**: `same_assign` 이 `True` 라는 것 — **군집화 결과는 한 글자도 안 바뀌고** 붙는 단어만 달라집니다. 표현 모델은 '묶기'가 아니라 '이름표 달기' 단계이기 때문입니다.
- **어느 쪽을 골라야 하나**: 정답은 없습니다. 교안의 뉴스 데이터에서는 c-TF-IDF 가 더 읽혔고, 리뷰 데이터에서는 다를 수 있습니다. **두 결과를 눈으로 비교해 고르는 것**이 이 문제의 목적입니다 — "최신이니까 좋다" 는 근거가 되지 못합니다.
- **흔한 실수**: `topic_model` 에 `representation_model` 을 대입만 하고 다시 학습하지 않는 것. 키워드는 **학습할 때** 만들어지므로 새 모델로 `fit_transform` 해야 합니다.

## 9. LLM 에게 토픽 이름 맡기기 (조합)
**배경**: 문제 3~7 에서 토픽의 정체는 **키워드를 사람이 읽어** 파악했습니다. 이 마지막 한 걸음도 자동화할 수 있습니다 — BERTopic 의 **`representation_model`** 에 LLM 을 끼우면, 학습하면서 토픽마다 이름을 지어 `get_topic_info()` 의 **`Name` 컬럼**에 담아 줍니다(교안 맛보기 절).

**요구사항**:
- 문제 3 과 **똑같은 표준 구성**(`min_cluster_size=25`)으로 모델을 하나 더 만들되, `representation_model` 에 아래 LLM 표현 모델을 끼워 `named_model` 에 담고 학습하세요.
```
from openai import OpenAI
from bertopic.representation import OpenAI as OpenAIRepresentation

prompt = ('다음은 한국어 상품 리뷰 토픽입니다.\n'
          '대표 리뷰:\n[DOCUMENTS]\n'
          '키워드: [KEYWORDS]\n'
          '이 토픽의 이름을 한국어 10자 이내로 하나만 답하세요. 설명 없이 이름만.')
rep_model = OpenAIRepresentation(client=OpenAI(), model='gpt-4o-mini',
                                 chat=True, nr_docs=4, prompt=prompt)
```
- 학습 후 `named_model.get_topic_info()` 를 `named_info` 에 담고, 문제 3 의 `topic_info` 와 **`Name` 컬럼을 나란히** 출력해 무엇이 달라졌는지 보세요.
- LLM 이 지은 이름만 모아 리스트 `topic_names` 에 담으세요(노이즈 −1 제외).

**예시**
```
topic_info 의 Name   ->  0_착용_사이즈_느낌_불편   (키워드 나열)
named_info 의 Name   ->  0_자세교정 밴드           (LLM 이 지은 이름)
len(topic_names)     ->  2   (노이즈를 뺀 토픽 수만큼)
```

> **키가 없으면 건너뜁니다.** `../../day13_토픽모델링_심화/.env` 에 `OPENAI_API_KEY` 가 있으면 실제로 호출하고, 없으면 이 문제는 **채점에서 제외**됩니다(과금 없음). 키는 `.env.example` 을 복사해 채우세요.
<details><summary>힌트</summary>

```text
접근방법:
- 문제 3 의 구성을 그대로 만들되 representation_model 인자에 LLM 표현 모델을 넘긴다.
- 학습이 끝나면 토픽 정보 표의 Name 열이 곧 LLM 이 지은 이름이다.

세부구현:
1. .env 를 읽어 키가 없으면 이 문제를 건너뛴다
2. OpenAI 클라이언트와 OpenAIRepresentation 을 만든다(프롬프트는 지문 그대로)
3. 문제 3 과 같은 표준 구성에 representation_model 을 더해 named_model 을 만든다
4. fit_transform 에 docs 와 embeddings=emb 를 넘긴다
5. get_topic_info 를 named_info 에 담고 Topic 이 -1 이 아닌 행의 Name 을 topic_names 에 모은다
```

</details>

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv('../../day13_토픽모델링_심화/.env')

if not os.getenv('OPENAI_API_KEY'):
    print('OPENAI_API_KEY 가 없어 이 문제는 건너뜁니다(채점 제외).')
    named_info = None
    topic_names = []
else:
    from openai import OpenAI
    from bertopic.representation import OpenAI as OpenAIRepresentation

    prompt = ('다음은 한국어 상품 리뷰 토픽입니다.\n'
              '대표 리뷰:\n[DOCUMENTS]\n'
              '키워드: [KEYWORDS]\n'
              '이 토픽의 이름을 한국어 10자 이내로 하나만 답하세요. 설명 없이 이름만.')
    rep_model = OpenAIRepresentation(client=OpenAI(), model='gpt-4o-mini',
                                     chat=True, nr_docs=4, prompt=prompt)

    named_umap = UMAP(n_components=5, n_neighbors=15, min_dist=0.0, metric='cosine', random_state=42)
    named_hdbscan = HDBSCAN(min_cluster_size=25, metric='euclidean',
                            prediction_data=True)
    named_vectorizer = CountVectorizer(tokenizer=korean_tokenizer, max_df=0.85)
    named_model = BERTopic(embedding_model=emb_model, umap_model=named_umap,
                           hdbscan_model=named_hdbscan, vectorizer_model=named_vectorizer,
                           representation_model=rep_model, language='multilingual', verbose=False)
    named_model.fit_transform(docs, embeddings=emb)

    named_info = named_model.get_topic_info()
    topic_names = named_info.loc[named_info['Topic'] != -1, 'Name'].tolist()

    compare = pd.DataFrame({'키워드 표현': topic_info.loc[topic_info['Topic'] != -1, 'Name'].tolist(),
                            'LLM 이 지은 이름': topic_names})
    display(compare)
    print('같은 토픽인데 이름이 사람 말로 바뀌었다 — 리포트에 그대로 쓸 수 있다')

In [ ]:
# [자가채점]
import os
if not os.getenv('OPENAI_API_KEY'):
    print('⏭️ 키가 없어 문제9 는 채점에서 제외합니다.')
else:
    _n = int((topic_info['Topic'] != -1).sum())
    assert named_info is not None, 'named_info 가 없습니다 — 모델을 학습했는지 확인하세요'
    assert len(topic_names) == _n, f'토픽 이름은 노이즈를 뺀 토픽 수({_n})만큼 나와야 합니다'
    assert all(isinstance(x, str) and x.strip() for x in topic_names), '이름이 비어 있습니다'
    # 키워드 나열이 아니라 LLM 이 지은 이름인지 — 원래 Name 과 달라야 한다
    _orig = topic_info.loc[topic_info['Topic'] != -1, 'Name'].tolist()
    assert topic_names != _orig, 'Name 이 그대로입니다 — representation_model 이 실제로 끼워졌는지 확인하세요'
    print('✅ 문제9 통과! 지어진 이름:', topic_names)

### 해설 — 문제 9
- **접근법**: BERTopic 의 조립 구조가 여기서 다시 드러납니다 — UMAP·HDBSCAN·CountVectorizer 를 끼웠던 것과 **똑같은 방식**으로 `representation_model` 자리에 LLM 을 끼웁니다. 나머지 코드는 한 줄도 바뀌지 않습니다.
- **무엇이 달라졌나**: `Name` 컬럼이 `0_착용_사이즈_느낌_불편`(키워드 나열)에서 `0_자세교정 밴드` 같은 **사람이 읽는 이름**으로 바뀝니다. 토픽을 *찾는* 일은 여전히 BERTopic 이 하고, LLM 은 마지막 **해석·명명**만 맡습니다.
- **흔한 실수**: 1) 문제 3 의 `topic_model` 에 `representation_model` 을 나중에 대입하고 다시 학습하지 않는 것 — 이름은 **학습할 때** 만들어지므로 새로 `fit_transform` 해야 합니다. 2) 키를 코드에 직접 적는 것 — 반드시 `.env` 에 두고 `load_dotenv` 로 읽습니다(키는 공유·커밋 금지).
- **비용**: 토픽 수만큼만 호출합니다(여기선 2회). 토픽이 수십 개면 호출도 그만큼 늘어나니 `nr_docs` 와 토픽 수를 먼저 확인하고 돌리세요.